# Kalman Historical V3 — Return + Regime

V2를 고정 benchmark로 두고 V3 alpha 구조를 평가합니다.\n
- US: Ridge forward-return model\n- KR: ElasticNet forward-return model\n- BTC: HistGradientBoosting forward-return model\n- Separate regime gate\n- Inner validation objective: return / Sharpe proxy / turnover\n- V2 vs V3 common-window replay\n- Equal Weight portfolio only\n- Qlib recorder: Python 3.12 isolated environment\n
**Safety:** Research only / Toss OFF / Neon write OFF\n

In [ ]:
from google.colab import drive
import json
import shutil
import subprocess
import traceback
from datetime import datetime
from pathlib import Path

PINNED_SHA = "6d5598594e101c03abe5780654c99ecf22bd7ccc"
SOURCE_BRANCH = "feature/historical-v3-return-regime-20260913"
V1_RUN_TAG = "20260913_042850"
V2_CANDIDATE_TAG = "20260913_nested_v2_001"
V3_CANDIDATE_TAG = "20260913_return_regime_v3_001"

drive.mount('/content/drive', force_remount=False)

diag_dir = Path('/content/drive/MyDrive/Kalman_Diagnostics')
diag_dir.mkdir(parents=True, exist_ok=True)
bootstrap = diag_dir / 'historical_v3_return_regime_bootstrap_status.json'

def write_status(status, **extra):
    payload = {
        'status': status,
        'updated_at': datetime.now().astimezone().isoformat(),
        'pinned_sha': PINNED_SHA,
        'v3_candidate_tag': V3_CANDIDATE_TAG,
        **extra,
    }
    tmp = bootstrap.with_suffix('.json.tmp')
    tmp.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, default=str) + '\n',
        encoding='utf-8',
    )
    tmp.replace(bootstrap)

write_status('RUNNING', phase='CLONE')
try:
    repo = Path('/content/Codex')
    if repo.exists():
        shutil.rmtree(repo)

    subprocess.run(
        [
            'git', 'clone', '--branch', SOURCE_BRANCH,
            'https://github.com/kimtk94/Codex.git', str(repo),
        ],
        check=True,
    )
    subprocess.run(
        ['git', '-C', str(repo), 'checkout', '--detach', PINNED_SHA],
        check=True,
    )
    checked = subprocess.check_output(
        ['git', '-C', str(repo), 'rev-parse', 'HEAD'],
        text=True,
    ).strip()
    assert checked == PINNED_SHA, (checked, PINNED_SHA)

    app = repo / 'kalman-toss-gateway'
    runner = app / 'scripts' / 'colab_historical_v3_return_regime.py'
    required = [
        runner,
        app / 'research' / 'quant_stack' / 'historical_v3_return_regime.py',
        app / 'research' / 'quant_stack' / 'historical_v3_qlib.py',
        app / 'config' / 'model-v3-historical-return-regime-spec.json',
    ]
    missing = [str(path) for path in required if not path.exists()]
    assert not missing, missing

    write_status('RUNNING', phase='PY_COMPILE')
    subprocess.run(
        ['python', '-m', 'py_compile', *[str(p) for p in required if p.suffix == '.py']],
        check=True,
    )

    write_status('RUNNING', phase='V3_RUNNER')
    subprocess.run(
        [
            'python', str(runner),
            '--drive-root', '/content/drive/MyDrive',
            '--v1-run-tag', V1_RUN_TAG,
            '--v2-candidate-tag', V2_CANDIDATE_TAG,
            '--v3-candidate-tag', V3_CANDIDATE_TAG,
            '--pinned-code-sha', PINNED_SHA,
        ],
        check=True,
    )

    summary = (
        Path('/content/drive/MyDrive/Market_Model_V2/historical_quant_2017_v3_candidate')
        / V3_CANDIDATE_TAG
        / 'historical_v3_candidate_summary.json'
    )
    assert summary.exists(), summary
    write_status('COMPLETE', phase='DONE', summary=str(summary))
    print('\nSUMMARY:', summary)
    print(summary.read_text(encoding='utf-8'))
except Exception as exc:
    write_status(
        'FAIL',
        phase='BOOTSTRAP_OR_RUNNER',
        error_type=type(exc).__name__,
        error=str(exc),
        traceback=traceback.format_exc(),
    )
    raise
